# Notebook 32 — Final Law Extraction for the Universality Boundary

This notebook turns the dimensionless boundary analysis into a compact final law.

Main goals:
1. reuse the extracted boundary points,
2. identify the simplest acceptable law in each control direction,
3. write the final law in baseline-normalized form,
4. compare prediction against extracted boundary points,
5. summarize the result in the cleanest equation-level form.

The focus is not on more scanning. It is on extracting the most compact statement supported by the data.


In [ ]:
# Ensure dependencies are installed before imports
import importlib, subprocess, sys

def ensure(pkg):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

for pkg in ['qutip', 'numpy', 'matplotlib', 'scipy']:
    ensure(pkg)


## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from scipy.optimize import minimize_scalar
from qutip import basis, qeye, tensor, sigmax, mesolve


## Basis states and operators

In [ ]:
g = basis(2, 0)
r = basis(2, 1)
I = qeye(2)

sx = sigmax()
n = r * r.dag()
sigma_minus = g * r.dag()

gg = tensor(g, g)
gr = tensor(g, r)
rg = tensor(r, g)
rr = tensor(r, r)

basis_states = [gg, gr, rg, rr]

sx1 = tensor(sx, I)
sx2 = tensor(I, sx)
n1 = tensor(n, I)
n2 = tensor(I, n)
sm1 = tensor(sigma_minus, I)
sm2 = tensor(I, sigma_minus)

U_cz = np.diag([1, 1, 1, -1]).astype(complex)


## Baseline protocol and control grids

In [ ]:
baseline = {
    'T': 26.0,
    'alpha': 0.10,
    'omega_max': 1.09,
    'delta0': 1.00,
    'p': 1.00,
    'q': 0.72,
}
V = 4.0

T_vals = np.array([20.0, 24.0, 26.0, 30.0, 34.0])
alpha_vals = np.array([0.06, 0.08, 0.10, 0.12, 0.14])
omega_vals = np.array([0.95, 1.02, 1.09, 1.16, 1.23])

T0 = baseline['T']
alpha0 = baseline['alpha']
omega0 = baseline['omega_max']


## Shaped schedules and Hamiltonian

In [ ]:
def omega_shaped(t, T, omega_max, p):
    s = t / T
    base = 16.0 * (s * (1.0 - s)) ** 2
    return omega_max * np.maximum(base, 0.0) ** p

def delta_shaped(t, T, V, alpha, delta0, q):
    s = t / T
    shaped = s ** q
    return delta0 + alpha * V * 0.5 * (1.0 - np.cos(np.pi * shaped))

H_omega = 0.5 * (sx1 + sx2)
H_delta = -(n1 + n2)
H_V = n1 * n2

def build_time_dependent_hamiltonian(T, omega_max, alpha, V, delta0, p, q):
    def coeff_omega(t, args=None):
        return omega_shaped(t, T=T, omega_max=omega_max, p=p)
    def coeff_delta(t, args=None):
        return delta_shaped(t, T=T, V=V, alpha=alpha, delta0=delta0, q=q)
    return [
        [H_omega, coeff_omega],
        [H_delta, coeff_delta],
        [H_V, lambda t, args=None: V],
    ]


## Collapse operators

In [ ]:
def collapse_ops(gamma_decay=0.0, gamma_phi=0.0):
    ops = []
    if gamma_decay > 0:
        ops.append(np.sqrt(gamma_decay) * sm1)
        ops.append(np.sqrt(gamma_decay) * sm2)
    if gamma_phi > 0:
        ops.append(np.sqrt(gamma_phi) * n1)
        ops.append(np.sqrt(gamma_phi) * n2)
    return ops


## Noisy evolution and effective map

In [ ]:
def run_noisy_evolution(psi0, params, gamma_decay=0.0, gamma_phi=0.0, n_steps=140):
    H = build_time_dependent_hamiltonian(
        params['T'], params['omega_max'], params['alpha'], V, params['delta0'], params['p'], params['q']
    )
    times = np.linspace(0.0, params['T'], n_steps)
    c_ops = collapse_ops(gamma_decay=gamma_decay, gamma_phi=gamma_phi)
    result = mesolve(H, psi0, times, c_ops)
    return result.states[-1]

def state_to_column_mixed(state):
    vals = []
    for basis_state in basis_states:
        if state.isket:
            vals.append(basis_state.overlap(state))
        else:
            val = basis_state.dag() * state * basis_state
            vals.append(np.real(val.full()[0, 0]) if hasattr(val, 'full') else np.real(val))
    return np.array(vals, dtype=complex)

def build_noisy_effective_map(params, gamma_decay=0.0, gamma_phi=0.0, n_steps=140):
    cols = []
    finals = []
    for psi0 in basis_states:
        final_state = run_noisy_evolution(
            psi0, params, gamma_decay=gamma_decay, gamma_phi=gamma_phi, n_steps=n_steps
        )
        cols.append(state_to_column_mixed(final_state))
        finals.append(final_state)
    return np.column_stack(cols), finals


## Diagnostics

In [ ]:
def process_fidelity(U_eff, U_target):
    d = U_target.shape[0]
    return float(np.abs(np.trace(np.conjugate(U_target.T) @ U_eff)) ** 2 / (d ** 2))

def wrapped_phase(x):
    return np.angle(np.exp(1j * x))

def diagonal_phases(U_eff):
    return np.angle(np.diag(U_eff))

def entangling_phase(U_eff):
    phi = diagonal_phases(U_eff)
    return wrapped_phase(phi[0] + phi[3] - phi[1] - phi[2])

def extract_local_z_phases(U_eff):
    phi00, phi01, phi10, phi11 = diagonal_phases(U_eff)
    phi_ent = entangling_phase(U_eff)
    a = wrapped_phase(phi10 - phi00)
    b = wrapped_phase(phi01 - phi00)
    global_phase = phi00
    return global_phase, a, b, phi_ent

def local_z_diagonal(a, b):
    return np.diag([1.0, np.exp(1j*b), np.exp(1j*a), np.exp(1j*(a+b))]).astype(complex)

def compensate_local_z(U_eff):
    global_phase, a, b, phi_ent = extract_local_z_phases(U_eff)
    U1 = np.exp(-1j * global_phase) * U_eff
    U2 = U1 @ local_z_diagonal(-a, -b)
    return U2, global_phase, a, b, phi_ent

def compensated_cz_fidelity(U_eff):
    U_comp, _, _, _, _ = compensate_local_z(U_eff)
    return process_fidelity(U_comp, U_cz)

def leakage_from_finals(final_states):
    scores = []
    for idx, final_state in enumerate(final_states):
        basis_state = basis_states[idx]
        if final_state.isket:
            score = np.abs(basis_state.overlap(final_state)) ** 2
        else:
            val = basis_state.dag() * final_state * basis_state
            score = np.real(val.full()[0, 0]) if hasattr(val, 'full') else np.real(val)
        scores.append(score)
    return float(1.0 - np.mean(scores))

def coherence_proxy(final_states):
    vals = []
    for state in final_states:
        rho = state.proj() if state.isket else state
        arr = rho.full()
        off = arr.copy()
        np.fill_diagonal(off, 0.0)
        vals.append(np.mean(np.abs(off)))
    return float(np.mean(vals))


## Shared noise grid

In [ ]:
gamma_decay_vals = np.linspace(0.0, 0.12, 13)
gamma_phi_vals = np.linspace(0.0, 0.12, 13)


## Evaluate one protocol

In [ ]:
def evaluate_protocol(params):
    cz_map = np.zeros((len(gamma_phi_vals), len(gamma_decay_vals)))
    coh_map = np.zeros((len(gamma_phi_vals), len(gamma_decay_vals)))
    leak_map = np.zeros((len(gamma_phi_vals), len(gamma_decay_vals)))

    for i, gamma_phi in enumerate(gamma_phi_vals):
        for j, gamma_decay in enumerate(gamma_decay_vals):
            U_eff, finals = build_noisy_effective_map(
                params, gamma_decay=gamma_decay, gamma_phi=gamma_phi, n_steps=140
            )
            cz_map[i, j] = compensated_cz_fidelity(U_eff)
            coh_map[i, j] = coherence_proxy(finals)
            leak_map[i, j] = leakage_from_finals(finals)

    S = cz_map * coh_map * (1.0 - leak_map)
    S_norm = S / np.max(S) if np.max(S) > 0 else S.copy()

    S_gamma = S_norm[0, :]
    S_phi = S_norm[:, 0]
    interp_gamma = interp1d(gamma_decay_vals, S_gamma, kind='linear', fill_value='extrapolate')

    def loss(lam):
        mapped = lam * gamma_phi_vals
        pred = interp_gamma(np.clip(mapped, gamma_decay_vals.min(), gamma_decay_vals.max()))
        return float(np.mean((pred - S_phi) ** 2))

    fit = minimize_scalar(loss, bounds=(0.1, 5.0), method='bounded')
    lam = float(fit.x)
    axis_loss = float(fit.fun)

    gamma_eff_grid = np.zeros_like(S_norm)
    for i, gamma_phi in enumerate(gamma_phi_vals):
        for j, gamma_decay in enumerate(gamma_decay_vals):
            gamma_eff_grid[i, j] = gamma_decay + lam * gamma_phi

    gamma_eff_flat = gamma_eff_grid.ravel()
    S_flat = S_norm.ravel()
    order = np.argsort(gamma_eff_flat)
    ge = gamma_eff_flat[order]
    sv = S_flat[order]

    n_bins = 12
    bins = np.linspace(ge.min(), ge.max(), n_bins + 1)
    centers = 0.5 * (bins[:-1] + bins[1:])
    means = np.full(n_bins, np.nan)

    for k in range(n_bins):
        mask = (ge >= bins[k]) & (ge < bins[k+1])
        if np.any(mask):
            means[k] = np.mean(sv[mask])

    valid = ~np.isnan(means)
    x = centers[valid]
    y = means[valid]

    return {
        'params': params,
        'lambda': lam,
        'axis_loss': axis_loss,
        'curve_x': x,
        'curve_y': y,
    }


## Baseline reference and universality score

In [ ]:
baseline_res = evaluate_protocol(baseline)
lambda0 = baseline_res['lambda']
curve0_x = baseline_res['curve_x']
curve0_y = baseline_res['curve_y']
interp0 = interp1d(curve0_x, curve0_y, kind='linear', fill_value='extrapolate')

def curve_mismatch_to_baseline(x, y):
    y_ref = interp0(np.clip(x, curve0_x.min(), curve0_x.max()))
    return float(np.mean((y - y_ref) ** 2))

def universality_score(lambda_shift, axis_loss, curve_mismatch,
                       lambda_scale=0.15, loss_scale=0.001, curve_scale=0.001):
    return float(np.exp(
        -(lambda_shift / lambda_scale)
        -(axis_loss / loss_scale)
        -(curve_mismatch / curve_scale)
    ))

boundary_level = 0.30
print("baseline lambda =", lambda0)
print("boundary level =", boundary_level)


## Score maps

In [ ]:
TA_score = np.zeros((len(alpha_vals), len(T_vals)))
TO_score = np.zeros((len(omega_vals), len(T_vals)))

for i, alpha in enumerate(alpha_vals):
    for j, T in enumerate(T_vals):
        params = {**baseline, 'T': float(T), 'alpha': float(alpha)}
        res = evaluate_protocol(params)
        lambda_shift = abs(res['lambda'] - lambda0)
        curve_mismatch = curve_mismatch_to_baseline(res['curve_x'], res['curve_y'])
        TA_score[i, j] = universality_score(lambda_shift, res['axis_loss'], curve_mismatch)

for i, omega in enumerate(omega_vals):
    for j, T in enumerate(T_vals):
        params = {**baseline, 'T': float(T), 'omega_max': float(omega)}
        res = evaluate_protocol(params)
        lambda_shift = abs(res['lambda'] - lambda0)
        curve_mismatch = curve_mismatch_to_baseline(res['curve_x'], res['curve_y'])
        TO_score[i, j] = universality_score(lambda_shift, res['axis_loss'], curve_mismatch)


## Extract boundary points

In [ ]:
def extract_vertical_boundary(score_map, xvals, yvals, level):
    bx, by = [], []
    for i, y in enumerate(yvals):
        row = score_map[i, :]
        crossing = None
        for j in range(len(xvals) - 1):
            v0, v1 = row[j], row[j + 1]
            if (v0 >= level and v1 < level) or (v0 > level and v1 <= level):
                x0, x1 = xvals[j], xvals[j + 1]
                xc = x0 if v1 == v0 else x0 + (level - v0) * (x1 - x0) / (v1 - v0)
                crossing = xc
                break
        if crossing is None and np.max(row) >= level and np.min(row) >= level:
            crossing = float(xvals[-1])
        if crossing is not None:
            bx.append(float(crossing))
            by.append(float(y))
    return np.array(bx), np.array(by)

TA_bx, TA_by = extract_vertical_boundary(TA_score, T_vals, alpha_vals, boundary_level)
TO_bx, TO_by = extract_vertical_boundary(TO_score, T_vals, omega_vals, boundary_level)

TA_x = TA_by / alpha0
TA_y = TA_bx / T0

TO_x = TO_by / omega0
TO_y = TO_bx / T0

print("dimensionless T-alpha boundary:", list(zip(TA_x, TA_y)))
print("dimensionless T-omega boundary:", list(zip(TO_x, TO_y)))


## Final-law candidate fits

In [ ]:
def fit_constant(x, y):
    c = float(np.mean(y))
    yhat = np.full_like(y, c, dtype=float)
    return {'name': 'constant', 'coeffs': [c], 'yhat': yhat}

def fit_shifted_linear(x, y):
    coeffs = np.polyfit(x - 1.0, y, deg=1)
    yhat = np.polyval(coeffs, x - 1.0)
    return {'name': 'shifted_linear', 'coeffs': coeffs.tolist(), 'yhat': yhat}

def fit_linear(x, y):
    coeffs = np.polyfit(x, y, deg=1)
    yhat = np.polyval(coeffs, x)
    return {'name': 'linear', 'coeffs': coeffs.tolist(), 'yhat': yhat}

def score_fit(y, yhat):
    mse = float(np.mean((y - yhat) ** 2))
    mae = float(np.mean(np.abs(y - yhat)))
    var = float(np.var(y))
    r2 = float(1.0 - mse / var) if var > 0 else np.nan
    return {'mse': mse, 'mae': mae, 'r2': r2}

def choose_best_fits(x, y):
    fits = []
    for fitter in [fit_constant, fit_shifted_linear, fit_linear]:
        fit = fitter(x, y)
        fit.update(score_fit(y, fit['yhat']))
        fits.append(fit)
    return sorted(fits, key=lambda d: d['mse'])

TA_fits = choose_best_fits(TA_x, TA_y) if len(TA_x) >= 3 else []
TO_fits = choose_best_fits(TO_x, TO_y) if len(TO_x) >= 3 else []

best_TA = TA_fits[0] if TA_fits else None
best_TO = TO_fits[0] if TO_fits else None

print("best T_c(alpha)/T0 fit:", best_TA)
print("best T_c(omega_max)/T0 fit:", best_TO)


## Final-law plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

if best_TA is not None:
    axes[0].scatter(TA_x, TA_y, label='boundary points')
    xfine = np.linspace(TA_x.min(), TA_x.max(), 200)
    coeffs = best_TA['coeffs']
    if best_TA['name'] == 'constant':
        yfine = np.full_like(xfine, coeffs[0], dtype=float)
    elif best_TA['name'] == 'shifted_linear':
        yfine = np.polyval(coeffs, xfine - 1.0)
    else:
        yfine = np.polyval(coeffs, xfine)
    axes[0].plot(xfine, yfine, label=f"best: {best_TA['name']}")
    axes[0].axvline(1.0, linestyle='--', alpha=0.6)
    axes[0].axhline(1.0, linestyle='--', alpha=0.6)
    axes[0].set_xlabel('alpha / alpha0')
    axes[0].set_ylabel('T_c / T0')
    axes[0].set_title('Final law for T_c(alpha)')
    axes[0].legend()

if best_TO is not None:
    axes[1].scatter(TO_x, TO_y, label='boundary points')
    xfine = np.linspace(TO_x.min(), TO_x.max(), 200)
    coeffs = best_TO['coeffs']
    if best_TO['name'] == 'constant':
        yfine = np.full_like(xfine, coeffs[0], dtype=float)
    elif best_TO['name'] == 'shifted_linear':
        yfine = np.polyval(coeffs, xfine - 1.0)
    else:
        yfine = np.polyval(coeffs, xfine)
    axes[1].plot(xfine, yfine, label=f"best: {best_TO['name']}")
    axes[1].axvline(1.0, linestyle='--', alpha=0.6)
    axes[1].axhline(1.0, linestyle='--', alpha=0.6)
    axes[1].set_xlabel('omega_max / omega0')
    axes[1].set_ylabel('T_c / T0')
    axes[1].set_title('Final law for T_c(omega_max)')
    axes[1].legend()

plt.tight_layout()
plt.show()


## Residual checks for the chosen laws

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

if best_TA is not None:
    resid = TA_y - best_TA['yhat']
    axes[0].axhline(0.0, linestyle='--', alpha=0.7)
    axes[0].scatter(TA_x, resid)
    axes[0].set_xlabel('alpha / alpha0')
    axes[0].set_ylabel('residual in T_c / T0')
    axes[0].set_title(f"T_c(alpha): {best_TA['name']} residuals")

if best_TO is not None:
    resid = TO_y - best_TO['yhat']
    axes[1].axhline(0.0, linestyle='--', alpha=0.7)
    axes[1].scatter(TO_x, resid)
    axes[1].set_xlabel('omega_max / omega0')
    axes[1].set_ylabel('residual in T_c / T0')
    axes[1].set_title(f"T_c(omega_max): {best_TO['name']} residuals")

plt.tight_layout()
plt.show()


## Final extracted equations

In [ ]:
def format_constant(fit):
    return f"T_c / T0 ≈ {fit['coeffs'][0]:.6f}"

def format_shifted_linear(fit, varname):
    a, b = fit['coeffs']
    return f"T_c / T0 ≈ {a:.6f} * ({varname} - 1) + {b:.6f}"

def format_linear(fit, varname):
    a, b = fit['coeffs']
    return f"T_c / T0 ≈ {a:.6f} * {varname} + {b:.6f}"

def fit_equation_string(fit, varname):
    if fit['name'] == 'constant':
        return format_constant(fit)
    elif fit['name'] == 'shifted_linear':
        return format_shifted_linear(fit, varname)
    else:
        return format_linear(fit, varname)

eq_alpha = fit_equation_string(best_TA, "alpha/alpha0") if best_TA is not None else "n/a"
eq_omega = fit_equation_string(best_TO, "omega_max/omega0") if best_TO is not None else "n/a"

print("Final extracted law for alpha:")
print(eq_alpha)
print()
print("Final extracted law for omega_max:")
print(eq_omega)


## Compact summary

In [ ]:
lines = []
lines.append("Final law extraction")
lines.append("")
lines.append(f"Baseline values: T0={T0:.4f}, alpha0={alpha0:.4f}, omega0={omega0:.4f}")
lines.append(f"Boundary score level = {boundary_level:.2f}")
lines.append("")

if best_TA is not None:
    lines.append("Best law for T_c(alpha):")
    lines.append("  " + eq_alpha)
    lines.append(
        f"  fit = {best_TA['name']}, MSE={best_TA['mse']:.6e}, MAE={best_TA['mae']:.6e}, R2={best_TA['r2']:.6f}"
    )

if best_TO is not None:
    lines.append("")
    lines.append("Best law for T_c(omega_max):")
    lines.append("  " + eq_omega)
    lines.append(
        f"  fit = {best_TO['name']}, MSE={best_TO['mse']:.6e}, MAE={best_TO['mae']:.6e}, R2={best_TO['r2']:.6f}"
    )

lines.append("")
lines.append("Interpretation:")
lines.append("- A constant alpha law means alpha weakly deforms the universality boundary near baseline.")
lines.append("- A sloped omega law means omega_max is the dominant local boundary-shaping control.")
lines.append("- This is the compact equation-level summary of the extracted control-space boundary.")

summary_text = "\n".join(lines)
print(summary_text)


## Final conclusion

This notebook converts the dimensionless universality-boundary data into a final compact law.

That gives the cleanest project-level summary:

**near the shaped baseline, the universality boundary is approximately flat in alpha and approximately linear in omega_max, with the precise coefficients extracted directly from the data.**
